# RepairScope-X — Notebook 02 (FIXED v4)
## SDLM-3B real generation + trajectory + suffix-repair audit

**Fully standalone.** No ZIP, previous notebook, or project folder is required.

This version handles the three compatibility differences seen between the
official SDLM remote code and current Colab/Transformers:

1. missing `config.pad_token_id`;
2. old `_tied_weights_keys` list vs the new Transformers dictionary format;
3. old `DynamicCache.get_usable_length()` API.

The model class is dynamically imported and patched **before**
`from_pretrained()` constructs the model.

The actual experiment remains unchanged:
real SDLM generation → NFE/VRAM/trajectory audit → deterministic verifier →
exact-prefix-preserving suffix regeneration.

## Before running

Use **Runtime → Restart session** once before running this notebook because
the previous failed model construction may have left GPU/CPU objects alive.

The downloaded checkpoint should normally remain in the Hugging Face cache.

In [ ]:
!pip -q install -U accelerate sentencepiece psutil

In [ ]:
import os, sys, gc, json, time, random, platform, subprocess, shutil
from pathlib import Path
import numpy as np
import torch
import transformers

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "A CUDA GPU is required."

props = torch.cuda.get_device_properties(0)
GPU_GB = props.total_memory / 2**30

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated now:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if torch.cuda.memory_allocated()/2**30 > 2.0:
    raise RuntimeError(
        "More than 2 GB is already allocated. Restart the Colab runtime, "
        "then run this notebook from the top."
    )

## 1. Patch the current cache API for SDLM's older model code

In [ ]:
from transformers.cache_utils import DynamicCache

CACHE_SHIM_INSTALLED = False

if not hasattr(DynamicCache, "get_usable_length"):
    def _compat_get_usable_length(self, new_seq_length, layer_idx=0):
        previous = self.get_seq_length(layer_idx)

        max_len = None

        if hasattr(self, "get_max_cache_shape"):
            try:
                shape = self.get_max_cache_shape(layer_idx)
            except TypeError:
                try:
                    shape = self.get_max_cache_shape()
                except Exception:
                    shape = None
            except Exception:
                shape = None

            if isinstance(shape, int):
                max_len = shape
            elif shape is not None:
                try:
                    # Typical cache shapes are [..., seq_len, head_dim].
                    max_len = shape[-2] if len(shape) >= 2 else shape[-1]
                except Exception:
                    max_len = None

        if max_len is None and hasattr(self, "get_max_length"):
            try:
                max_len = self.get_max_length()
            except Exception:
                max_len = None

        # DynamicCache is normally unbounded.
        if max_len is None or max_len == -1:
            return previous

        if previous + new_seq_length > max_len:
            return max(0, max_len - new_seq_length)

        return previous

    DynamicCache.get_usable_length = _compat_get_usable_length
    CACHE_SHIM_INSTALLED = True

print("DynamicCache shim installed:", CACHE_SHIM_INSTALLED)
print("get_usable_length available:",
      hasattr(DynamicCache, "get_usable_length"))

## 2. Clone the official SDLM repository

The notebook uses the official `SDLM_generate` implementation and records
the Git commit.

In [ ]:
REPO_DIR = Path("/content/SDLM_official")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "https://github.com/OpenGVLab/SDLM.git",
        str(REPO_DIR)
    ],
    check=True
)

SDLM_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

sys.path.insert(0, str(REPO_DIR))
from sdlm_inference import SDLM_generate

print("Official SDLM commit:", SDLM_COMMIT)

## 3. Load and patch the remote configuration

In [ ]:
from transformers import AutoTokenizer, AutoConfig

MODEL_ID = "OpenGVLab/SDLM-3B-D4"
DTYPE = torch.float16

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

print("Remote config class:", type(config).__name__)
print("tie_word_embeddings:", getattr(config, "tie_word_embeddings", None))
print("tokenizer.pad_token_id:", tokenizer.pad_token_id)
print("config.eos_token_id:", getattr(config, "eos_token_id", None))
print("config.bos_token_id:", getattr(config, "bos_token_id", None))

PAD_ID = tokenizer.pad_token_id
if PAD_ID is None:
    PAD_ID = getattr(config, "eos_token_id", None)
if PAD_ID is None:
    PAD_ID = getattr(config, "bos_token_id", None)
if PAD_ID is None:
    PAD_ID = 151643

config.pad_token_id = int(PAD_ID)

# Explicit compatibility fields that newer Transformers may query.
defaults = {
    "decoder_start_token_id": None,
    "is_encoder_decoder": False,
    "add_cross_attention": False,
    "tie_encoder_decoder": False,
}

for k, v in defaults.items():
    if not hasattr(config, k):
        setattr(config, k, v)

print("Patched pad_token_id:", config.pad_token_id)

## 4. Dynamically import and patch the SDLM model class

New Transformers versions define `_tied_weights_keys` as a mapping:

`target_parameter -> source_parameter`

The SDLM/Qwen2 remote code follows the older convention:

`["lm_head.weight"]`

When the checkpoint configuration enables tied word embeddings, the
corresponding standard causal-LM mapping is:

`lm_head.weight -> model.embed_tokens.weight`

If word embeddings are not tied, the mapping is simply empty.

In [ ]:
from transformers.dynamic_module_utils import get_class_from_dynamic_module

auto_map = getattr(config, "auto_map", None)
assert isinstance(auto_map, dict), "Remote config has no auto_map."

class_ref = auto_map.get("AutoModelForCausalLM")
assert class_ref is not None, (
    "Checkpoint does not advertise an AutoModelForCausalLM remote class."
)

print("Remote model class reference:", class_ref)

ModelClass = get_class_from_dynamic_module(
    class_ref,
    MODEL_ID
)

original_tied = getattr(ModelClass, "_tied_weights_keys", None)

print("Original _tied_weights_keys:", original_tied)
print("Type:", type(original_tied).__name__)

TIED_KEYS_PATCHED = False

if isinstance(original_tied, list):
    if bool(getattr(config, "tie_word_embeddings", False)):
        # SDLM's Qwen2 CausalLM exposes the standard `lm_head` and
        # `model.embed_tokens` names.
        if original_tied == ["lm_head.weight"] or "lm_head.weight" in original_tied:
            ModelClass._tied_weights_keys = {
                "lm_head.weight": "model.embed_tokens.weight"
            }
        else:
            raise RuntimeError(
                "Unexpected legacy tied-weight list: "
                f"{original_tied}. Refusing to guess a mapping."
            )
    else:
        # New Transformers will not use tying when the config says false,
        # but converting the legacy list to the expected type avoids
        # version-specific initialization paths.
        ModelClass._tied_weights_keys = {}

    TIED_KEYS_PATCHED = True

elif original_tied is None:
    ModelClass._tied_weights_keys = {}
    TIED_KEYS_PATCHED = True

elif not isinstance(original_tied, dict):
    raise TypeError(
        "_tied_weights_keys has an unsupported type: "
        f"{type(original_tied).__name__}"
    )

print("Patched _tied_weights_keys:", ModelClass._tied_weights_keys)
print("Tied-key patch applied:", TIED_KEYS_PATCHED)

## 5. Construct the model through the patched class

In [ ]:
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

print("Loading", MODEL_ID)
print("dtype:", DTYPE)

t0 = time.time()

model = ModelClass.from_pretrained(
    MODEL_ID,
    config=config,
    trust_remote_code=True,
    attn_implementation="eager",
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to("cuda").eval()

LOAD_SECONDS = time.time() - t0

print("\nMODEL LOAD: PASS")
print("Model class:", type(model).__name__)
print("Device:", model.device)
print("Load time:", round(LOAD_SECONDS, 2), "s")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")
print("Peak:", round(torch.cuda.max_memory_allocated()/2**30, 3), "GB")
print("pad_token_id:", model.config.pad_token_id)

if bool(getattr(model.config, "tie_word_embeddings", False)):
    try:
        tied_physically = (
            model.lm_head.weight.data_ptr()
            == model.model.embed_tokens.weight.data_ptr()
        )
        print("Embedding/lm_head physically tied:", tied_physically)
    except Exception as e:
        print("Tie verification unavailable:", repr(e))

## 6. Actual forward-call counter

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = self.module.register_forward_pre_hook(self._hook)
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 7. Exact-verifier arithmetic prompt

In [ ]:
PROMPT = (
    "A shop has 12 boxes with 13 pens in each box. "
    "How many pens are there in total? "
    "Show brief reasoning and give the final numeric answer."
)

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Solve the problem carefully."
    },
    {"role": "user", "content": PROMPT},
]

chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    [chat_text],
    return_tensors="pt"
).to(model.device)

print("Prompt tokens:", model_inputs.input_ids.shape[1])

## 8. KV-cache smoke test

Cached inference is attempted first. If another cache-version incompatibility
appears, this capability notebook continues with cache disabled and records it.

In [ ]:
def run_sdlm(inputs, max_gen_len, use_cache, save_history=True):
    with ForwardCounter(model) as fc:
        t0 = time.time()

        result = SDLM_generate(
            model,
            tokenizer,
            inputs,
            max_gen_len=max_gen_len,
            temperature=0.0,
            threshold=0.5,
            n_future_tokens=4,
            alg="prob_conf",
            save_history=save_history,
            use_cache=use_cache,
        )

        elapsed = time.time() - t0

    return result, fc.calls, elapsed

CACHE_OK = False
CACHE_ERROR = None

try:
    torch.cuda.empty_cache()

    smoke, smoke_nfe, smoke_time = run_sdlm(
        model_inputs,
        max_gen_len=24,
        use_cache=True,
        save_history=True
    )

    CACHE_OK = True

    print("KV CACHE SMOKE: PASS")
    print("Smoke NFE:", smoke_nfe)
    print("Smoke time:", round(smoke_time, 3), "s")

except Exception as e:
    CACHE_ERROR = f"{type(e).__name__}: {e}"

    print("KV CACHE SMOKE: FAILED")
    print(CACHE_ERROR)
    print("\nContinuing with use_cache=False.")

    torch.cuda.empty_cache()
    gc.collect()

print("CACHE_OK =", CACHE_OK)

## 9. Real SDLM generation

In [ ]:
USE_CACHE = CACHE_OK

MAX_GEN_LEN = 128
THRESHOLD = 0.5
N_FUTURE_TOKENS = 4
ALG = "prob_conf"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with ForwardCounter(model) as fc:
    t0 = time.time()

    response, history = SDLM_generate(
        model,
        tokenizer,
        model_inputs,
        max_gen_len=MAX_GEN_LEN,
        temperature=0.0,
        threshold=THRESHOLD,
        n_future_tokens=N_FUTURE_TOKENS,
        alg=ALG,
        save_history=True,
        use_cache=USE_CACHE,
    )

    WALL_SECONDS = time.time() - t0

ACTUAL_NFE = fc.calls
PEAK_VRAM_GB = torch.cuda.max_memory_allocated()/2**30

print("\n========== RESPONSE ==========")
print(response[0])
print("==============================")
print("Cache used:", USE_CACHE)
print("Actual NFE:", ACTUAL_NFE)
print("Wall time:", round(WALL_SECONDS, 3), "s")
print("Peak VRAM:", round(PEAK_VRAM_GB, 3), "GB")
print("History entries:", len(history))

## 10. SDLM accepted-block trajectory

In [ ]:
import pandas as pd

trajectory = []
previous_len = 0

for step_idx, item in enumerate(history):
    decoded_batch, total_len = item
    total_len = int(total_len)
    accepted = total_len - previous_len
    previous_len = total_len

    trajectory.append({
        "iteration": step_idx + 1,
        "accepted_tokens": accepted,
        "total_generated_tokens": total_len,
        "text_so_far": decoded_batch[0],
    })

traj_df = pd.DataFrame(trajectory)

if len(traj_df):
    display(
        traj_df[
            ["iteration", "accepted_tokens", "total_generated_tokens"]
        ].head(40)
    )

    TRAJECTORY_SUMMARY = {
        "iterations": int(len(traj_df)),
        "total_history_tokens":
            int(traj_df.total_generated_tokens.iloc[-1]),
        "mean_accepted_tokens":
            float(traj_df.accepted_tokens.mean()),
        "median_accepted_tokens":
            float(traj_df.accepted_tokens.median()),
        "max_accepted_tokens":
            int(traj_df.accepted_tokens.max()),
        "fraction_single_token_iterations":
            float((traj_df.accepted_tokens == 1).mean()),
        "tokens_per_actual_forward":
            float(
                traj_df.total_generated_tokens.iloc[-1]
                / max(ACTUAL_NFE, 1)
            ),
    }
else:
    TRAJECTORY_SUMMARY = {}

print(json.dumps(TRAJECTORY_SUMMARY, indent=2))

## 11. Deterministic numeric verifier

In [ ]:
import re, math

def extract_last_number(text):
    values = re.findall(
        r"[-+]?(?:\d*\.)?\d+(?:[eE][-+]?\d+)?",
        str(text).replace(",", "")
    )
    return float(values[-1]) if values else None

def verify_numeric(text, target):
    pred = extract_last_number(text)

    passed = (
        pred is not None
        and math.isclose(
            pred,
            float(target),
            abs_tol=1e-9,
            rel_tol=1e-9,
        )
    )

    return {
        "passed": bool(passed),
        "prediction": pred,
        "target": float(target),
    }

VERIFICATION = verify_numeric(response[0], 156)
print(VERIFICATION)

## 12. Faithful SDLM repair primitive

SDLM is sequential/block diffusion, so this notebook tests:

**preserve exact prompt + response prefix → regenerate suffix**

It does not claim arbitrary middle-span repair.

In [ ]:
from transformers import BatchEncoding

original_response = response[0]

response_ids = tokenizer(
    original_response,
    add_special_tokens=False,
    return_tensors="pt"
).input_ids[0]

cut = max(1, len(response_ids)//2)
preserved_ids = response_ids[:cut].to(model.device)

prompt_ids = model_inputs.input_ids[0]

repair_input_ids = torch.cat(
    [prompt_ids, preserved_ids]
).unsqueeze(0)

repair_inputs = BatchEncoding({
    "input_ids": repair_input_ids,
    "attention_mask": torch.ones_like(repair_input_ids),
})

preserved_text = tokenizer.decode(
    preserved_ids,
    skip_special_tokens=False
)

print("Original output tokens:", len(response_ids))
print("Preserved output tokens:", cut)
print("\nPRESERVED PREFIX:")
print(preserved_text)

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with ForwardCounter(model) as fc:
    t0 = time.time()

    new_suffix, repair_history = SDLM_generate(
        model,
        tokenizer,
        repair_inputs,
        max_gen_len=96,
        temperature=0.0,
        threshold=THRESHOLD,
        n_future_tokens=N_FUTURE_TOKENS,
        alg=ALG,
        save_history=True,
        use_cache=USE_CACHE,
    )

    REPAIR_WALL = time.time() - t0

REPAIR_NFE = fc.calls
REPAIR_PEAK_GB = torch.cuda.max_memory_allocated()/2**30

repaired_full = preserved_text + new_suffix[0]

print("\n========== ORIGINAL ==========")
print(original_response)
print("\n========== PREFIX + NEW SUFFIX ==========")
print(repaired_full)
print("\nRepair NFE:", REPAIR_NFE)
print("Repair time:", round(REPAIR_WALL, 3), "s")
print("Repair peak VRAM:", round(REPAIR_PEAK_GB, 3), "GB")

## 13. Exact preservation and capability matrix

In [ ]:
reconstructed_prefix = repair_input_ids[0, -cut:].detach().cpu()
original_prefix = preserved_ids.detach().cpu()

assert torch.equal(reconstructed_prefix, original_prefix)

print("Prefix token preservation: 100%")
print("SUFFIX REPAIR PRIMITIVE: PASS")

CAPABILITIES = {
    "model": MODEL_ID,
    "architecture": "sequential/block diffusion",
    "arbitrary_middle_span_repair_validated": False,
    "prefix_preserving_suffix_regeneration": True,
    "fresh_regeneration": True,
    "history_available": True,
    "accepted_block_sizes_available": True,
    "whole_response_token_flip_trajectory": False,
    "kv_cache_compatible_current_runtime": bool(CACHE_OK),
    "config_pad_token_id": int(model.config.pad_token_id),
    "dynamic_cache_shim_installed": bool(CACHE_SHIM_INSTALLED),
    "legacy_tied_weight_keys_patched": bool(TIED_KEYS_PATCHED),
}

print(json.dumps(CAPABILITIES, indent=2))

## 14. Save result locally

In [ ]:
SAVE_DIR = Path("/content/RepairScopeX_Notebook02_v4")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RESULT = {
    "model_id": MODEL_ID,
    "sdlm_git_commit": SDLM_COMMIT,
    "python": platform.python_version(),
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_gb": GPU_GB,
    "pad_token_id": int(model.config.pad_token_id),
    "tied_key_patch": bool(TIED_KEYS_PATCHED),
    "cache_shim": bool(CACHE_SHIM_INSTALLED),
    "cache_ok": bool(CACHE_OK),
    "cache_error": CACHE_ERROR,
    "generation": {
        "prompt": PROMPT,
        "response": response[0],
        "verification": VERIFICATION,
        "actual_nfe": ACTUAL_NFE,
        "wall_seconds": WALL_SECONDS,
        "peak_vram_gb": PEAK_VRAM_GB,
        "trajectory_summary": TRAJECTORY_SUMMARY,
    },
    "repair": {
        "preserved_tokens": int(cut),
        "new_suffix": new_suffix[0],
        "actual_nfe": REPAIR_NFE,
        "wall_seconds": REPAIR_WALL,
        "peak_vram_gb": REPAIR_PEAK_GB,
    },
    "capabilities": CAPABILITIES,
}

result_path = SAVE_DIR/"notebook02_result.json"

result_path.write_text(
    json.dumps(
        RESULT,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

if len(traj_df):
    traj_df.drop(
        columns=["text_so_far"],
        errors="ignore"
    ).to_csv(
        SAVE_DIR/"trajectory.csv",
        index=False
    )

print("Saved:", result_path)

# Success criteria

The important checkpoints are:

```
MODEL LOAD: PASS
```

followed eventually by:

```
Prefix token preservation: 100%
SUFFIX REPAIR PRIMITIVE: PASS
```

`CACHE_OK=False` is acceptable for this capability notebook; it only means
its wall-clock time cannot be used for official SDLM efficiency comparisons.